# Long-Running Loop Optimization
### AME-UP WIL Project – Tartigrade

**Author:** Shabiba  
**Focus:** Multiprocessing & batching for CPU-bound workloads  

## Objective
Analyze how a long-running, independent loop can be optimized using Python multiprocessing, and evaluate the impact of task batching under constrained environments.


In [18]:
import time
import math

def heavy_compute(x):
    total = 0
    for i in range(2_000_000):
        total += math.sqrt(x + i)
    return total

def run_sequential(n_tasks):
    results = []
    start = time.time()
    for i in range(n_tasks):
        results.append(heavy_compute(i))
    end = time.time()
    return end - start


In [19]:
runtime = run_sequential(50)
print(f"Sequential runtime: {runtime:.2f} seconds")

Sequential runtime: 11.20 seconds


### Baseline (Sequential Execution)

- Execution model: Single process
- Characteristics:
  - Independent iterations
  - CPU-bound work
- Observed runtime: 11.20 seconds for 50 tasks

This confirms the workload is suitable for parallelization.

## Report Outline

1. Problem Description
   - Nature of long-running loop
   - Why it does not scale sequentially

2. Baseline Approach
   - Sequential execution
   - Limitations

3. Multiprocessing Strategy
   - Task independence
   - Process pool design
   - Batching approach

4. Performance Comparison
   - Runtime differences
   - CPU utilization discussion

5. Trade-offs & Constraints
   - Overhead
   - Memory
   - Colab limitations

6. Recommendations
   - When to use multiprocessing
   - When to consider distributed systems

In [20]:
from multiprocessing import Pool, cpu_count

In [21]:
def run_multiprocessing(n_tasks):
    start = time.time()

    with Pool(processes=cpu_count()) as pool:
        results = pool.map(heavy_compute, range(n_tasks))

    end = time.time()
    return end - start

In [22]:
seq_time = run_sequential(50)
mp_time = run_multiprocessing(50)

print(f"Sequential time: {seq_time:.2f} seconds")
print(f"Multiprocessing time: {mp_time:.2f} seconds")

Sequential time: 11.27 seconds
Multiprocessing time: 11.89 seconds


### Environment Constraint

Google Colab is not ideal for benchmarking multiprocessing due to limited CPU access and high process-spawn overhead. Results here demonstrate conceptual behavior rather than absolute performance gains.

In [23]:
def run_multiprocessing_batched(n_tasks, batch_size):
    start = time.time()

    batches = [
        list(range(i, min(i + batch_size, n_tasks)))
        for i in range(0, n_tasks, batch_size)
    ]

    with Pool(processes=cpu_count()) as pool:
        pool.map(process_batch, batches)

    end = time.time()
    return end - start

In [24]:
def process_batch(batch):
    return [heavy_compute(i) for i in batch]

In [25]:
print("Sequential:", run_sequential(50))
print("MP (no batching):", run_multiprocessing(50))
print("MP (batch size = 5):", run_multiprocessing_batched(50, 5))
print("MP (batch size = 10):", run_multiprocessing_batched(50, 10))

Sequential: 11.196629524230957
MP (no batching): 11.55115556716919
MP (batch size = 5): 10.922439098358154
MP (batch size = 10): 11.651984930038452


### Batching Experiment Results

Increasing task count to 50 revealed that naive multiprocessing remained slower than sequential execution due to overhead. Introducing batching with a moderate batch size (5 tasks per batch) resulted in improved performance by reducing inter-process communication. Larger batch sizes reduced parallelism and led to performance degradation.

This demonstrates the trade-off between task granularity and resource utilization.
